## 00 - Project Introduction

We will build a Transformer that reads a news article and picks one of four
categories: World, Sports, Business, or Sci/Tech. The input is plain text. The
model outputs four scores (**logits**), one for each category. At prediction
time we can turn the scores into probabilities.

```text
News text -> word IDs with [CLS] -> embeddings + position information
          -> two Transformer Encoder blocks -> final [CLS] vector
          -> linear classifier -> four class logits
```

An **Encoder** is enough because we want to understand the input, not generate
an output sentence. We are not building a decoder, a translation system, or a
text generator. "From scratch" means we write the attention and encoder
operations and train fresh weights. PyTorch still provides tensors, automatic
differentiation, basic layers, and the optimizer. We use no pretrained weights
or tokenizers.

## 01 - Environment Setup

Run this notebook from top to bottom. The first code cell installs the
required libraries, including PyTorch 2.1 or newer if needed. It does not
request upgrades, so a compatible preinstalled PyTorch build (such as
Colab's CUDA build) stays in place.

Edit the settings in the next Python cell. Quick mode uses 6,000 training
articles, 1,000 validation articles, and two epochs. Turn `FAST_MODE` off to
use the entire 90/10 training/validation partition of the official training
data and train for five epochs. CPU works; a Colab GPU is recommended for the
larger run or the optional experiments. A fixed seed helps reproducibility,
but results can still differ between environments and devices.

In [ ]:
%pip install -q "torch>=2.1" "datasets>=2.18" "matplotlib>=3.7" "numpy>=1.24" "scikit-learn>=1.2"

In [ ]:
import csv
import random
import re
import time
from collections import Counter
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Change the run settings here, then rerun the notebook from this section.
SEED = 42
FAST_MODE = True
RUN_BONUS_EXPERIMENTS = False
USE_POSITIONAL_ENCODING = True
OUTPUT_DIR = Path("outputs")

VALIDATION_FRACTION = 0.10
FAST_TRAIN_SIZE = 6000
FAST_VAL_SIZE = 1000
MIN_WORD_FREQUENCY = 2
MAX_LENGTH = 64  # Includes the [CLS] slot.

BATCH_SIZE = 64
EPOCHS = 2 if FAST_MODE else 5
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01

D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.1


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # Seeds PyTorch on CPU and CUDA.


if NUM_HEADS < 1 or D_MODEL % NUM_HEADS != 0:
    raise ValueError("D_MODEL must be divisible by NUM_HEADS.")
if RUN_BONUS_EXPERIMENTS and D_MODEL % 4 != 0:
    raise ValueError("D_MODEL must be divisible by 4 to compare 1 and 4 heads.")

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    # Keep the quick run responsive without increasing the thread count.
    torch.set_num_threads(min(4, torch.get_num_threads()))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
mode_name = "quick" if FAST_MODE else "full"
print(f"Mode: {mode_name} | Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}")
print(f"Run bonus experiments: {RUN_BONUS_EXPERIMENTS}")

## 02 - Load and Explore the Dataset

[AG News](https://huggingface.co/datasets/fancyzhx/ag_news) downloads
automatically. Each row has article `text` and one integer `label`:

| Class ID | Category | Typical stories |
|---|---|---|
| 0 | World | International politics and events |
| 1 | Sports | Games, teams, and athletes |
| 2 | Business | Companies, markets, and the economy |
| 3 | Sci/Tech | Science, computing, and new technology |

We first split the official training rows into disjoint 90% training and 10%
validation partitions. **Stratified** means that each partition preserves
roughly the same mix of labels. Quick mode then takes a stratified sample
from *each* partition. Full mode keeps both partitions in their entirety.
Training learns weights and the vocabulary; validation helps choose the
epoch to keep. Neither job uses the official test rows. We report only the
test size here and leave test articles and labels for Section 12.

In [ ]:
def make_development_indices(labels, fast_mode=FAST_MODE, seed=SEED):
    labels = np.asarray(labels)
    row_indices = np.arange(len(labels))
    train_indices, val_indices = train_test_split(
        row_indices, test_size=VALIDATION_FRACTION, stratify=labels, random_state=seed
    )

    if fast_mode:
        train_indices, _ = train_test_split(
            train_indices,
            train_size=FAST_TRAIN_SIZE,
            stratify=labels[train_indices],
            random_state=seed,
        )
        val_indices, _ = train_test_split(
            val_indices,
            train_size=FAST_VAL_SIZE,
            stratify=labels[val_indices],
            random_state=seed,
        )

    return train_indices, val_indices


dataset = load_dataset("fancyzhx/ag_news")
official_train = dataset["train"]
label_names = list(official_train.features["label"].names)

train_indices, val_indices = make_development_indices(official_train["label"])
train_rows = official_train.select(train_indices.tolist())
validation_rows = official_train.select(val_indices.tolist())

print(f"Official training rows: {len(official_train):,}")
print(f"Selected training rows: {len(train_rows):,}")
print(f"Selected validation rows: {len(validation_rows):,}")
print(f"Held-out official test rows: {dataset.num_rows['test']:,}")
print("Label order:", list(enumerate(label_names)))

print("\nOne selected training example per class:")
shown_labels = set()
for row in train_rows:
    label = row["label"]
    if label not in shown_labels:
        print(f"\n{label} ({label_names[label]}): {row['text']}")
        shown_labels.add(label)
    if len(shown_labels) == len(label_names):
        break

## 03 - Build a Simple Tokenizer

We lowercase English text and treat each run of letters or digits as one
token; punctuation separates words. A vocabulary maps words to integer IDs
for the embedding lookup. We count words **only in the selected training
rows**, keeping words that appear at least twice by default. Words that
are rare or unseen become `[UNK]`, even during validation or testing.

- `[PAD]` (0) fills unused slots on the right so every sequence in a batch
  has the same length.
- `[UNK]` (1) stands for a word that is not in the vocabulary. It is still a
  real input to attention.
- `[CLS]` (2) goes at the start, even for empty text. The classifier reads
  its final vector. We keep at most `max_length - 1` words to leave it room.

The model's padding mask is `True` for `[CLS]` and every real token, including
`[UNK]`, and `False` for `[PAD]`. Query, Key, and Value (Q, K, and V) are not
three datasets. Later, three learned projections make them from the same
input representations inside each self-attention layer.

For inference from a checkpoint in a fresh session, run Section 01 and the
first code cell below to define preprocessing without loading AG News. Use
the checkpoint's vocabulary instead of building a new one.

In [ ]:
PAD_ID, UNK_ID, CLS_ID = 0, 1, 2


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def build_vocabulary(texts, min_frequency=MIN_WORD_FREQUENCY):
    word_counts = Counter()
    for text in texts:
        word_counts.update(tokenize(text))

    vocabulary = {"[PAD]": PAD_ID, "[UNK]": UNK_ID, "[CLS]": CLS_ID}
    for word, count in word_counts.most_common():
        if count < min_frequency:
            break
        vocabulary[word] = len(vocabulary)
    return vocabulary


def encode_text(text, vocabulary, max_length):
    if max_length < 1:
        raise ValueError("Leave at least one slot for [CLS].")

    words = tokenize(text)[: max_length - 1]
    token_ids = [CLS_ID]
    for word in words:
        token_ids.append(vocabulary.get(word, UNK_ID))
    token_ids.extend([PAD_ID] * (max_length - len(token_ids)))
    return token_ids


def make_tensor_dataset(rows, vocabulary, max_length):
    sequences = []
    labels = []
    for row in rows:
        sequences.append(encode_text(row["text"], vocabulary, max_length))
        labels.append(row["label"])

    # Each example pairs a length-T sequence of token IDs with one class ID.
    input_tensor = torch.tensor(sequences, dtype=torch.long)
    label_tensor = torch.tensor(labels, dtype=torch.long)
    return TensorDataset(input_tensor, label_tensor)


def make_loader(data, shuffle=False, seed=SEED):
    # Give each fit its own seeded shuffle instead of reusing a loader's state.
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator)

In [ ]:
vocab = build_vocabulary(train_rows["text"])
train_data = make_tensor_dataset(train_rows, vocab, MAX_LENGTH)
validation_data = make_tensor_dataset(validation_rows, vocab, MAX_LENGTH)
val_loader = make_loader(validation_data)

print(f"Vocabulary size (including special tokens): {len(vocab):,}")
print(f"Training inputs: {tuple(train_data.tensors[0].shape)} [examples, tokens]")
print(f"Training labels: {tuple(train_data.tensors[1].shape)} [examples]")

In [ ]:
# Use a word outside this vocabulary to make the unknown-token behavior visible.
unknown_word = "madeupword"
while unknown_word in vocab:
    unknown_word += "x"

example_text = f"The Market opened today: {unknown_word}!"
example_ids = encode_text(example_text, vocab, max_length=8)
example_mask = [token_id != PAD_ID for token_id in example_ids]
print("Original sentence:", example_text)
print("Lowercase:", example_text.lower())
print("Tokens:", tokenize(example_text))
print("Token IDs (8 slots for this demo):", example_ids)
print("Valid-token mask:", example_mask)
print("Same sentence truncated to 4 slots:", encode_text(example_text, vocab, 4))

## 04 - Word Embeddings and Positional Encoding

The tokenizer gives us integer IDs shaped `[batch_size, sequence_length]`.
An embedding looks up a trainable vector of length `d_model` for each ID.
The result is `[batch_size, sequence_length, d_model]`. The `[PAD]` entry
starts at zero and is not learned. We still need an attention mask to keep
padding out of the computation for real tokens.

The embedding for a word is the same at every position. To tell the encoder
where each token occurs, we add a fixed vector for that position when
`USE_POSITIONAL_ENCODING` is enabled. In each pair of columns `2i` and `2i+1`,
we take the sine and cosine of the same angle:

$$
\text{angle}(\text{position}, i) = \frac{\text{position}}{10000^{2i / d_{\text{model}}}}
$$

Different column pairs change at different rates. The table has shape
`[1, max_length, d_model]`; the leading dimension lets every example in the
batch use the same positions. It is a **buffer**: it moves with the model to
CPU or CUDA, but the optimizer does not train it. With positions turned off,
the encoder has no way to tell whether two real words trade places.

In [ ]:
import math


class TokenAndPositionEmbeddings(nn.Module):
    def __init__(self, vocab_size, max_length, d_model, pad_id, use_positional_encoding):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.use_positional_encoding = use_positional_encoding

        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        even_columns = torch.arange(0, d_model, 2, dtype=torch.float32)
        frequencies = 1.0 / (10000.0 ** (even_columns / d_model))
        angles = positions * frequencies  # One row per position, one column per pair.

        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(angles)
        # If d_model is odd, the final sine column has no cosine partner.
        encoding[:, 1::2] = torch.cos(angles[:, : d_model // 2])
        self.register_buffer("position_encoding", encoding.unsqueeze(0))

    def forward(self, input_ids):
        token_vectors = self.token_embedding(input_ids)  # [B, T, d_model]
        if self.use_positional_encoding:
            sequence_length = input_ids.size(1)
            token_vectors = token_vectors + self.position_encoding[:, :sequence_length, :]
        return token_vectors

## 05 - Scaled Dot-Product Attention

For each token, a **query** says what to look for, a **key** says what can be
matched, and a **value** supplies the information to mix in. These are not
three datasets. We will compute them from the same token vectors using three
learned linear layers.

Below, `B` is batch size, `H` is the number of heads, `T` is sequence length,
and `head_dim` is the width of one head.

| Tensor | Shape | Meaning |
| --- | --- | --- |
| Queries, keys, values | `[B, H, T, head_dim]` each | One vector per token per head |
| Scores and weights | `[B, H, T, T]` each | Rows are query tokens; columns are key tokens |
| Context | `[B, H, T, head_dim]` | A weighted sum of values for each query |

$$
\operatorname{Attention}(Q,K,V) = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

With more features in a head, unscaled dot products can grow large and make
softmax too peaked. Dividing by $\sqrt{d_k}$, where $d_k$ is `head_dim`, helps
keep the scores in a useful range.

`valid_tokens` is Boolean `[B, T]`, with `True` for a real token. Set PAD
**key** scores to negative infinity **before** the softmax over the last
axis. Those columns get zero weight; the remaining weights still sum to one.
`[CLS]` is always present, so every row has at least one key to attend to,
even for empty text. PAD **queries** still produce rows of weights. That is
harmless: their vectors stay masked as keys in every block, and neither the
per-token operations nor the classifier read them into a real token's output.

In [ ]:
def scaled_dot_product_attention(queries, keys, values, valid_tokens):
    head_dim = queries.size(-1)
    scores = torch.matmul(queries, keys.transpose(-2, -1))  # [B, H, T, T]
    scores = scores / math.sqrt(head_dim)

    # Share the key mask across all heads and every query row.
    valid_keys = valid_tokens[:, None, None, :]  # [B, 1, 1, T]
    scores = scores.masked_fill(~valid_keys, float("-inf"))
    weights = torch.softmax(scores, dim=-1)

    context = torch.matmul(weights, values)  # [B, H, T, head_dim]
    return context, weights

## 06 - Multi-Head Self-Attention

Each projection starts with the same `[B, T, d_model]` input. We divide
its output features into `num_heads` groups of width
`head_dim = d_model // num_heads`, so `d_model` must divide evenly into the
heads. Separate groups can learn to attend to different token patterns.

Reshape from `[B, T, d_model]` to `[B, T, H, head_dim]`, then transpose the
token and head axes to get `[B, H, T, head_dim]`. This puts the token and
feature axes last for the matrix multiplications. After attention, undo these
two steps to get `[B, T, d_model]` again. A final linear layer mixes information
from the heads. Return the normalized weights as well so we can inspect them.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError("Choose positive d_model and num_heads with d_model divisible by num_heads.")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)

    def split_heads(self, projected):
        batch_size, sequence_length, _ = projected.shape
        heads = projected.reshape(batch_size, sequence_length, self.num_heads, self.head_dim)
        return heads.transpose(1, 2)  # [B, H, T, head_dim]

    def forward(self, token_vectors, valid_tokens):
        queries = self.split_heads(self.query(token_vectors))
        keys = self.split_heads(self.key(token_vectors))
        values = self.split_heads(self.value(token_vectors))
        context, weights = scaled_dot_product_attention(queries, keys, values, valid_tokens)

        batch_size, sequence_length, _ = token_vectors.shape
        context = context.transpose(1, 2)  # [B, T, H, head_dim]
        combined = context.reshape(batch_size, sequence_length, self.d_model)
        return self.output_projection(combined), weights

## 07 - Feed-Forward Network

Attention lets tokens share information. The feed-forward network then
processes each token separately with the same two linear layers. It expands
`[B, T, d_model]` to `[B, T, ff_dim]`, applies ReLU for a nonlinear
transformation, and projects back to `[B, T, d_model]`.

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.expand = nn.Linear(d_model, ff_dim)
        self.shrink = nn.Linear(ff_dim, d_model)

    def forward(self, token_vectors):
        expanded = self.expand(token_vectors)  # [B, T, ff_dim]
        activated = torch.relu(expanded)
        return self.shrink(activated)  # [B, T, d_model]

## 08 - Transformer Encoder Block

A block first applies self-attention, then the feed-forward network. In each
sublayer we keep a **residual connection**: an unchanged copy of its input is
added to the sublayer's output. During training, dropout acts on the sublayer
output, not on this shortcut. We apply layer normalization **after** each
addition (Post-LN). The shortcut preserves a direct path for information and
gradients; normalization keeps each token's features on a controlled scale.
Both operations act on `[B, T, d_model]`, and layer normalization works across
the features of each token, not across the sequence.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, num_heads)
        self.feed_forward = FeedForwardNetwork(d_model, ff_dim)
        self.attention_norm = nn.LayerNorm(d_model)
        self.feed_forward_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_vectors, valid_tokens):
        attention_output, weights = self.attention(token_vectors, valid_tokens)
        token_vectors = self.attention_norm(token_vectors + self.dropout(attention_output))

        feed_forward_output = self.feed_forward(token_vectors)
        token_vectors = self.feed_forward_norm(token_vectors + self.dropout(feed_forward_output))
        return token_vectors, weights

## 09 - Complete Transformer Encoder Classifier

We turn IDs into embeddings and pass them through `num_layers` encoder
blocks, each with its own parameters. The padding mask comes from the
original IDs once and stays the same across the blocks. Each input starts
with `[CLS]` at index zero. That token can gather information from the entire
real sequence, so we take its final `[B, d_model]` representation and map it to
`[B, num_classes]` raw logits (four scores per AG News article). We leave
softmax off the logits during training because cross-entropy takes raw scores.

With `return_attention=True`, the model also gives one `[B, H, T, T]` tensor
per encoder layer, in order from first to last.

In [ ]:
class TransformerEncoderClassifier(nn.Module):
    def __init__(
        self, vocab_size, max_length, num_classes, pad_id,
        d_model, num_heads, num_layers, ff_dim, dropout, use_positional_encoding,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.max_length = max_length
        self.embeddings = TokenAndPositionEmbeddings(
            vocab_size, max_length, d_model, pad_id, use_positional_encoding
        )
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, return_attention=False):
        if input_ids.ndim != 2 or not 1 <= input_ids.size(1) <= self.max_length:
            raise ValueError(f"Expected IDs shaped [batch_size, sequence_length] with 1 to {self.max_length} tokens.")

        valid_tokens = input_ids != self.pad_id  # [B, T]
        token_vectors = self.embeddings(input_ids)  # [B, T, d_model]
        attention_maps = []
        for layer in self.layers:
            token_vectors, weights = layer(token_vectors, valid_tokens)
            if return_attention:
                attention_maps.append(weights)

        cls_vectors = token_vectors[:, 0, :]  # [B, d_model]
        logits = self.classifier(cls_vectors)  # [B, num_classes]
        if return_attention:
            return logits, attention_maps
        return logits

## 10 - Model Sanity Checks

These are small, synthetic CPU checks, not AG News training or evaluation.
First compare attention against values calculated by hand, both on its own
and inside a two-head layer whose projection weights we set ourselves. Then
check shapes, masking, extra right padding, the positional switch, and
backpropagation. Finally, ask a fresh tiny model to memorize four different
inputs. Fitting this batch is a debugging check, not a measure of how well
the model generalizes.

In [ ]:
def check_attention_math():
    queries = torch.tensor([[1., 0., 0., 0.], [0., 1., 0., 0.], [0., 0., 0., 0.]])
    keys = torch.tensor([[2., 0., 0., 0.], [0., 4., 0., 0.], [200., 200., 0., 0.]])
    values = torch.tensor([[2., 0., 0., 0.], [0., 4., 0., 0.], [500., 500., 500., 500.]])
    valid_tokens = torch.tensor([[True, True, False]])
    # Add batch and head axes to the three [T, head_dim] matrices.
    context, weights = scaled_dot_product_attention(
        queries[None, None], keys[None, None], values[None, None], valid_tokens
    )

    # Divide by sqrt(4) = 2. After masking, the score pairs are [1, 0],
    # [0, 2], and [0, 0]; the third key gets no weight even for a PAD query.
    expected_weights = torch.tensor([
        [0.7310586, 0.2689414, 0.],
        [0.1192029, 0.8807971, 0.],
        [0.5, 0.5, 0.],
    ])
    expected_context = torch.tensor([
        [1.4621172, 1.0757656, 0., 0.],
        [0.2384058, 3.5231884, 0., 0.],
        [1., 2., 0., 0.],
    ])
    torch.testing.assert_close(weights[0, 0], expected_weights)
    torch.testing.assert_close(context[0, 0], expected_context)


def check_multi_head_attention():
    attention = MultiHeadSelfAttention(d_model=8, num_heads=2)
    identity = torch.eye(8)
    token_vectors = torch.tensor([[
        [1., 0., 0., 0., 0., 2., 0., 0.],
        [0., 2., 0., 0., 1., 0., 0., 0.],
        [100., 100., 0., 0., 100., 100., 0., 0.],
    ]])
    valid_tokens = torch.tensor([[True, True, False]])

    with torch.no_grad():
        # Q doubles the input and shifts feature 4; K copies it; V triples it.
        attention.query.weight.copy_(2 * identity)
        attention.query.bias.zero_()
        attention.query.bias[4] = 2
        attention.key.weight.copy_(identity)
        attention.key.bias.zero_()
        attention.value.weight.copy_(3 * identity)
        attention.value.bias.zero_()
        attention.output_projection.weight.copy_(2 * identity)
        attention.output_projection.bias.zero_()
        output, weights = attention(token_vectors, valid_tokens)

        changed_padding = token_vectors.clone()
        changed_padding[:, 2, :] = -300
        changed_output, _ = attention(changed_padding, valid_tokens)

    # Scale by sqrt(4), not sqrt(8). The two real queries have scaled,
    # unmasked score pairs [1, 0] and [0, 4] in head 0, [4, 1] and [0, 2] in head 1.
    expected_weights = torch.tensor([
        [[0.7310586, 0.2689414, 0.], [0.0179862, 0.9820138, 0.]],
        [[0.9525741, 0.0474259, 0.], [0.1192029, 0.8807971, 0.]],
    ])
    # Weight the values by these probabilities, join the heads, then double.
    expected_output = torch.tensor([
        [4.3863516, 3.2272968, 0., 0., 0.2845554, 11.4308892, 0., 0.],
        [0.1079172, 11.7841656, 0., 0., 5.2847826, 1.4304348, 0., 0.],
    ])
    torch.testing.assert_close(weights[0, :, :2, :], expected_weights)
    torch.testing.assert_close(output[0, :2, :], expected_output)
    torch.testing.assert_close(changed_output[:, :2, :], output[:, :2, :])

In [ ]:
def check_encoder_outputs(model, model_without_positions, input_ids):
    model.eval()
    model_without_positions.eval()
    more_padding = torch.zeros(4, 2, dtype=torch.long)
    padded_ids = torch.cat((input_ids, more_padding), dim=1)
    reordered_ids = input_ids[:, [0, 2, 1, 3]]  # Exchange two words, keeping CLS first.
    with torch.no_grad():
        logits, attention_maps = model(input_ids, return_attention=True)
        padded_logits = model(padded_ids)
        reordered_logits = model(reordered_ids)
        orderless_logits = model_without_positions(input_ids)
        orderless_reordered = model_without_positions(reordered_ids)

    assert logits.shape == (4, 4), "Expected one four-class logit vector per example."
    assert len(attention_maps) == 2, "Expected one attention map per encoder layer."
    for weights in attention_maps:
        assert weights.shape == (4, 3, 4, 4), "Expected [B, H, T, T] attention."
        row_sums = weights.sum(dim=-1)
        torch.testing.assert_close(row_sums, torch.ones_like(row_sums))
        pad_keys = (input_ids == 0)[:, None, None, :].expand_as(weights)
        torch.testing.assert_close(weights[pad_keys], torch.zeros_like(weights[pad_keys]))

    torch.testing.assert_close(padded_logits, logits, rtol=1e-5, atol=1e-6)
    # Same parameters in the two models: now only positions can break order invariance.
    torch.testing.assert_close(orderless_reordered, orderless_logits, rtol=1e-5, atol=1e-6)
    assert not torch.allclose(reordered_logits, logits), "Positions should make word order visible."
    print(f"Logits: {tuple(logits.shape)}; attention per layer: {tuple(attention_maps[0].shape)}")


def check_backward_and_fit(model, input_ids, labels):
    model.train()
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
    for step in range(40):
        optimizer.zero_grad()
        loss = loss_function(model(input_ids), labels)
        loss.backward()
        if step == 0:
            initial_loss = loss.item()
            for layer in model.layers:
                attention = layer.attention
                for name, projection in [("query", attention.query), ("key", attention.key), ("value", attention.value)]:
                    gradient = projection.weight.grad
                    assert gradient is not None, f"No gradient reached the {name} weights."
                    assert torch.isfinite(gradient).all(), f"Nonfinite gradient in the {name} weights."
                    assert gradient.abs().sum() > 0, f"Zero gradient in the {name} weights."
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(input_ids)
        final_loss = loss_function(logits, labels).item()
    assert torch.equal(logits.argmax(dim=1), labels), "The tiny batch was not memorized."
    assert final_loss < 0.1 and final_loss < initial_loss, "Expected a small final loss."
    print(f"Tiny batch fit: all 4 labels correct, loss {initial_loss:.3f} -> {final_loss:.3f}")


def run_encoder_sanity_checks():
    check_attention_math()
    check_multi_head_attention()
    print("Standalone and multi-head attention math passed.")

    # Use an odd d_model to exercise a positional table with an unpaired sine column.
    torch.manual_seed(42)
    config = dict(
        vocab_size=12, max_length=6, num_classes=4, pad_id=0,
        d_model=15, num_heads=3, num_layers=2, ff_dim=24,
        dropout=0.0, use_positional_encoding=True,
    )
    model = TransformerEncoderClassifier(**config)
    config["use_positional_encoding"] = False
    model_without_positions = TransformerEncoderClassifier(**config)
    model_without_positions.load_state_dict(model.state_dict())

    # The last example represents empty text: only CLS remains unmasked.
    input_ids = torch.tensor([[2, 3, 4, 0], [2, 5, 6, 0], [2, 7, 8, 0], [2, 0, 0, 0]])
    labels = torch.tensor([0, 1, 2, 3])
    check_encoder_outputs(model, model_without_positions, input_ids)
    check_backward_and_fit(model, input_ids, labels)
    print("Encoder sanity checks passed. Training will start with a new model.")


run_encoder_sanity_checks()

## 11 - Training

Each batch has token IDs of shape `[B, T]` and integer class IDs of shape
`[B]`. The model returns raw logits of shape `[B, 4]`. Column 0 is World,
column 1 is Sports, column 2 is Business, and column 3 is Sci/Tech: exactly
the dataset's class order. For example, label `2` means "use the Business
column as the correct answer for this article." `CrossEntropyLoss` uses
these class IDs to compare the correct scores against all four scores.
Give it raw logits, not softmax probabilities; the loss already applies
log-softmax internally.

AdamW updates the freshly initialized weights after each training batch.
After each epoch, validation uses evaluation mode (dropout off) without
computing gradients. We average loss per example so a short final batch
doesn't count as much as a full one. We restore the weights from the
lowest validation-loss epoch (the earlier epoch wins a tie), not
necessarily the last epoch. The test set does not decide which weights we
keep.

In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def evaluate(model, data_loader):
    model.eval()
    device = next(model.parameters()).device
    loss_function = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct_count = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for input_ids, labels in data_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            logits = model(input_ids)
            predictions = logits.argmax(dim=1)

            total_loss += loss_function(logits, labels).item() * labels.size(0)
            correct_count += (predictions == labels).sum().item()
            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    return {
        "loss": total_loss / len(all_labels),
        "accuracy": correct_count / len(all_labels),
        "labels": all_labels,
        "predictions": all_predictions,
    }


def train_model(config, training_data, validation_loader, seed=SEED, epochs=EPOCHS):
    set_seed(seed)
    model = TransformerEncoderClassifier(**config).to(DEVICE)
    training_loader = make_loader(training_data, shuffle=True, seed=seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_function = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    print(f"Fresh model: {count_parameters(model):,} trainable parameters")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for input_ids, labels in training_loader:
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids)  # [B, C] raw scores, one row per article.
            loss = loss_function(logits, labels)  # [B] correct class IDs.
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)

        train_loss = total_loss / len(training_data)
        validation = evaluate(model, validation_loader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(validation["loss"])
        history["val_accuracy"].append(validation["accuracy"])

        if validation["loss"] < best_loss:
            best_loss = validation["loss"]
            best_epoch = epoch
            # Copy the tensors so later optimizer steps cannot change this snapshot.
            best_state = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch}/{epochs} | train loss {train_loss:.4f} | "
            f"val loss {validation['loss']:.4f} | val accuracy {validation['accuracy']:.2%}"
        )

    if best_state is None:
        raise RuntimeError("No best epoch was found. Train for at least one epoch with finite loss.")
    model.load_state_dict(best_state)
    model.eval()
    print(f"Restored epoch {best_epoch} (lowest validation loss).")
    return model, history

In [ ]:
MODEL_CONFIG = {
    "vocab_size": len(vocab),
    "max_length": MAX_LENGTH,
    "num_classes": len(label_names),
    "pad_id": PAD_ID,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "ff_dim": FF_DIM,
    "dropout": DROPOUT,
    "use_positional_encoding": USE_POSITIONAL_ENCODING,
}

baseline_model, baseline_history = train_model(MODEL_CONFIG, train_data, val_loader)

## 12 - Evaluation

The training loop has already selected the model with the lowest **validation
loss**. Now we evaluate that fixed model on the full official test split, using
the vocabulary built from training text. We do not use the test results to
choose an epoch or change the model.

**Accuracy** is the fraction of correct predictions. The report breaks the
scores down by class: **precision** asks how often a prediction of that class
is correct, **recall** asks how many true examples were found, and **support**
is the number of true examples. **F1** combines precision and recall;
**macro F1** averages it across the four categories with equal weight.

In [ ]:
test_data = make_tensor_dataset(dataset["test"], vocab, MODEL_CONFIG["max_length"])
test_loader = make_loader(test_data, shuffle=False)
test_metrics = evaluate(baseline_model, test_loader)

test_labels = test_metrics["labels"]
test_predictions = test_metrics["predictions"]
class_ids = list(range(len(label_names)))
macro_f1 = f1_score(
    test_labels, test_predictions, labels=class_ids, average="macro", zero_division=0
)

print(f"Official test examples: {len(test_labels):,}")
print(f"Test accuracy: {test_metrics['accuracy']:.2%}")
print(f"Test macro F1: {macro_f1:.4f}")
print(classification_report(
    test_labels, test_predictions, labels=class_ids, target_names=label_names,
    zero_division=0,
))

In the loss plot, check whether validation loss falls along with training loss.
If training loss keeps falling while validation loss rises, the model may be
overfitting. In the confusion matrix, read across a row to see where examples
of one true class went. The diagonal counts correct predictions.

In [ ]:
epoch_numbers = range(1, len(baseline_history["train_loss"]) + 1)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.plot(epoch_numbers, baseline_history["train_loss"], marker="o", label="Training")
ax.plot(epoch_numbers, baseline_history["val_loss"], "--s", label="Validation")
ax.set(title="Loss during baseline training", xlabel="Epoch", ylabel="Mean cross-entropy")
ax.set_xticks(epoch_numbers)
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.savefig(OUTPUT_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
matrix = confusion_matrix(test_labels, test_predictions, labels=class_ids)
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
display_matrix = ConfusionMatrixDisplay(matrix, display_labels=label_names)
display_matrix.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set(title="Official test confusion matrix", xlabel="Predicted category", ylabel="True category")
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 13 - Test Your Own Text

Change the sentence below and rerun the example. Prediction uses the same
tokenizer, vocabulary, and length as training. Softmax turns the four logits
into probabilities that add up to one. The largest one picks the category,
but even a high probability does not guarantee a correct prediction.

In [ ]:
def predict(text, model=None, vocabulary=None, model_config=None, class_names=None):
    """Return the predicted category and a dictionary of all class probabilities."""
    if model is None:
        model = baseline_model
        vocabulary = vocab
        model_config = MODEL_CONFIG
        class_names = label_names
    elif vocabulary is None or model_config is None or class_names is None:
        raise ValueError("For a supplied model, pass its vocabulary, config, and class names.")

    token_ids = encode_text(text, vocabulary, model_config["max_length"])
    device = next(model.parameters()).device
    input_ids = torch.tensor([token_ids], dtype=torch.long, device=device)  # [1, sequence_length]
    model.eval()  # Turn off dropout for inference.
    with torch.no_grad():
        logits = model(input_ids)  # [1, number_of_classes]
        probabilities = torch.softmax(logits, dim=-1)[0].cpu().tolist()
        top_class_id = logits.argmax(dim=-1).item()

    class_probabilities = dict(zip(class_names, probabilities))
    return class_names[top_class_id], class_probabilities

In [ ]:
example_text = "Apple announced a new smartphone"
original_category, original_probabilities = predict(example_text)

print(f"Text: {example_text}")
print(f"Predicted category: {original_category}")
for name, probability in original_probabilities.items():
    print(f"  {name}: {probability:.2%}")

## 14 - Attention Visualization

Choose a **zero-based** layer and head below: index `0` means the first one.
Use a short sentence so the labels are easy to read. A row is a **query**
token; a column is a **key** token. A brighter square means the query gives
that key's value more weight in this head. Look for `[CLS]` on both axes and
`[UNK]` wherever a word was not in the vocabulary. The plot leaves out padded
rows and columns and shows only tokens that made it past truncation.

These are real weights from one head of the trained baseline, not a complete
explanation of the prediction. Other heads, the value vectors, and later
layers also affect the final logits.

In [ ]:
def plot_attention(text, layer_index=0, head_index=0):
    """Draw the learned attention for one layer and one head of the baseline."""
    if not 0 <= layer_index < MODEL_CONFIG["num_layers"]:
        raise ValueError(f"Choose a layer from 0 to {MODEL_CONFIG['num_layers'] - 1}.")
    if not 0 <= head_index < MODEL_CONFIG["num_heads"]:
        raise ValueError(f"Choose a head from 0 to {MODEL_CONFIG['num_heads'] - 1}.")

    token_ids = encode_text(text, vocab, MODEL_CONFIG["max_length"])
    input_ids = torch.tensor([token_ids], dtype=torch.long, device=DEVICE)
    baseline_model.eval()
    with torch.no_grad():
        _, layer_weights = baseline_model(input_ids, return_attention=True)

    # Padding is on the right, so the real tokens form one prefix on each axis.
    token_count = sum(token_id != MODEL_CONFIG["pad_id"] for token_id in token_ids)
    id_to_token = {token_id: token for token, token_id in vocab.items()}
    tokens = [id_to_token[token_id] for token_id in token_ids[:token_count]]
    # Each layer has weights [batch, head, query, key].
    weights = layer_weights[layer_index][0, head_index, :token_count, :token_count]

    figure_side = max(6, 0.45 * token_count)
    fig, ax = plt.subplots(figsize=(figure_side + 2, figure_side), layout="constrained")
    image = ax.imshow(weights.cpu().numpy(), cmap="viridis", vmin=0, vmax=1)
    positions = range(token_count)
    ax.set_xticks(positions, tokens, rotation=60, ha="right")
    ax.set_yticks(positions, tokens)
    ax.set(
        title=f"Baseline attention: layer {layer_index}, head {head_index}",
        xlabel="Key token (looked at)",
        ylabel="Query token (doing the looking)",
    )
    fig.colorbar(image, ax=ax, label="Attention weight")
    return fig

In [ ]:
attention_layer = 0
attention_head = 0
fig = plot_attention(example_text, layer_index=attention_layer, head_index=attention_head)
fig.savefig(OUTPUT_DIR / "attention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 15 - BONUS: Controlled Architecture Experiments

Set `RUN_BONUS_EXPERIMENTS = True` in Section 01 and rerun the notebook to do
four extra fits. The **reference** is trained fresh with four heads, positional
encoding on, and two layers. Each other run changes just one of these:

| Comparison | Reference | Variant |
|---|---|---|
| A. Attention heads | 4 heads | 1 head (same `d_model`, wider head) |
| B. Positional encoding | On | Off |
| C. Depth | 2 layers | 1 layer |

Every run starts with a new model and optimizer, resets the seed, and uses the
same train/validation rows, shuffled batch order, batch size, AdamW settings, and
epoch budget. We record scores on **validation only** after restoring the epoch
with the lowest validation loss. The baseline from Section 11 is not replaced.

In [ ]:
def run_architecture_experiment(name, config):
    print(f"\nTraining a fresh model: {name}")
    # Wait for queued GPU work so the timer measures this fit.
    on_cuda = torch.device(DEVICE).type == "cuda"
    if on_cuda:
        torch.cuda.synchronize(DEVICE)
    started_at = time.perf_counter()
    model, _ = train_model(config, train_data, val_loader, seed=SEED, epochs=EPOCHS)
    if on_cuda:
        torch.cuda.synchronize(DEVICE)
    train_seconds = time.perf_counter() - started_at

    metrics = evaluate(model, val_loader)
    result = {
        "experiment": name,
        **config,
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "device": str(DEVICE),
        "training_examples": len(train_data),
        "validation_examples": len(val_loader.dataset),
        "trainable_parameters": count_parameters(model),
        "val_loss": metrics["loss"],
        "val_accuracy": metrics["accuracy"],
        "train_seconds": train_seconds,
    }
    print(f"Validation loss: {result['val_loss']:.4f}; accuracy: {result['val_accuracy']:.2%}")
    print(f"Trainable parameters: {result['trainable_parameters']:,}; fit time: {train_seconds:.2f} s")
    return result

In [ ]:
if RUN_BONUS_EXPERIMENTS:
    model_width = MODEL_CONFIG["d_model"]
    if model_width % MODEL_CONFIG["num_heads"] != 0 or model_width % 4 != 0:
        raise ValueError("For the bonus, d_model must be divisible by NUM_HEADS and by 4.")

    reference = dict(MODEL_CONFIG, num_heads=4, use_positional_encoding=True, num_layers=2)
    experiment_configs = [
        ("Reference", reference),
        ("1 head", dict(reference, num_heads=1)),
        ("No positions", dict(reference, use_positional_encoding=False)),
        ("1 layer", dict(reference, num_layers=1)),
    ]
    experiment_results = []
    for name, config in experiment_configs:
        experiment_results.append(run_architecture_experiment(name, config))

    csv_path = OUTPUT_DIR / "architecture_experiments.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=list(experiment_results[0]))
        writer.writeheader()
        writer.writerows(experiment_results)

    names = [result["experiment"] for result in experiment_results]
    losses = [result["val_loss"] for result in experiment_results]
    accuracies = [result["val_accuracy"] for result in experiment_results]
    colors = ["#244f73", "#438678", "#438678", "#438678"]
    fig, (loss_ax, accuracy_ax) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")

    loss_bars = loss_ax.bar(names, losses, color=colors)
    loss_ax.set(title="Validation loss (lower is better)", ylabel="Mean cross-entropy")
    loss_ax.margins(y=0.16)
    loss_ax.bar_label(loss_bars, fmt="%.3f", padding=3)

    accuracy_bars = accuracy_ax.bar(names, accuracies, color=colors)
    accuracy_ax.set(title="Validation accuracy (higher is better)", ylabel="Fraction correct")
    accuracy_ax.set_ylim(0, 1.1)
    accuracy_ax.bar_label(accuracy_bars, fmt="%.3f", padding=3)

    fig.suptitle(f"Architecture comparisons on the same validation split (seed {SEED})")
    fig.savefig(OUTPUT_DIR / "architecture_experiments.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Bonus is off. Set RUN_BONUS_EXPERIMENTS = True in Section 01 to enable it.")

Read each variant against the reference, not against another variant that
changes a different setting. The CSV pairs scores with parameter counts and
fit times, including validation during training. Fewer layers also means
fewer parameters, so that comparison does not isolate depth from model
capacity. Equal epochs mean equal passes through the data, not equal compute
time. These are **exploratory, one-seed** results for this split and budget;
they do not prove that a design is better in general. Compare several seeds
before drawing a stronger conclusion, and do not use the test set to rank designs.

## 16 - Save and Reload

The checkpoint contains weights and the metadata needed to interpret them:
token IDs, model settings (including input length), and label order. It does
not store the Python class or tokenizer.

In Colab, open the **Files** sidebar, expand `outputs`, and use the menu next to
a file to **Download** it before the temporary runtime resets.

**To predict from a checkpoint in a fresh kernel**, run only these cells:

1. Section 01 (installation and setup).
2. The definition-only cell in Section 03, starting with `PAD_ID, UNK_ID, CLS_ID`.
3. The encoder definition cells in Sections 04–09.
4. The `predict` definition in Section 13, not its demonstration.
5. The `load_checkpoint` definition below, not the save or comparison cell.

Skip the dataset, vocabulary building, training, evaluation, and plots.
Upload your downloaded file or use a copy on a mounted drive. In a new code
cell, change `saved_path` to your file's path:

```python
saved_path = Path("transformer_checkpoint.pt")
model, vocabulary, config, class_names = load_checkpoint(saved_path)
category, probabilities = predict(
    "A new telescope mapped distant galaxies.", model, vocabulary, config, class_names
)
print(category, probabilities)
```

In a normal top-to-bottom run, the cells below save the trained baseline and
check the reloaded prediction against the one captured before the bonus fits.

In [ ]:
checkpoint_path = OUTPUT_DIR / "transformer_checkpoint.pt"
torch.save({
    "model_state_dict": baseline_model.state_dict(),
    "vocab": vocab,
    "model_config": MODEL_CONFIG,
    "label_names": label_names,
}, checkpoint_path)

In [ ]:
def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    restored_config = checkpoint["model_config"]
    restored_model = TransformerEncoderClassifier(**restored_config).to(DEVICE)
    restored_model.load_state_dict(checkpoint["model_state_dict"])
    restored_model.eval()
    return restored_model, checkpoint["vocab"], restored_config, checkpoint["label_names"]

In [ ]:
reloaded_model, reloaded_vocab, reloaded_config, reloaded_labels = load_checkpoint(checkpoint_path)
reloaded_category, reloaded_probabilities = predict(
    example_text,
    model=reloaded_model,
    vocabulary=reloaded_vocab,
    model_config=reloaded_config,
    class_names=reloaded_labels,
)

# Compare with the prediction saved in Section 13, before any bonus fits.
assert reloaded_category == original_category, "Reload changed the predicted category."
assert list(reloaded_probabilities) == list(original_probabilities), "Reload changed the label order."
for name, probability in original_probabilities.items():
    assert abs(reloaded_probabilities[name] - probability) < 1e-6, f"Reload changed {name} probability."

print(f"Reloaded prediction for {example_text!r}: {reloaded_category}")
for name, probability in reloaded_probabilities.items():
    print(f"  {name}: {probability:.2%}")
print("The category and all four probabilities match the original prediction within tolerance.")

## 17 - Final Educational Summary

- **What we built:** a regex tokenizer, token and position representations,
  masked multi-head self-attention, stacked encoder blocks, and a classifier
  that reads `[CLS]`.
- **What PyTorch did for us:** tensors and matrix operations, basic layers such
  as `Embedding`, `Linear`, `LayerNorm`, and `Dropout`, automatic gradients,
  cross-entropy, and AdamW updates. We assembled the architecture and wrote
  the attention math ourselves rather than calling a ready-made encoder.
- **What the model learned:** the weights started from scratch, without a
  pretrained model or vocabulary. News labels helped the model learn useful
  token embeddings, attention projections, and a mapping from the final
  `[CLS]` vector to class scores. Q, K, and V were computed inside the model,
  not supplied as training data.
- **Limits and next steps:** the word tokenizer loses some spelling and
  punctuation detail, maps unseen words to `[UNK]`, and cuts off long inputs.
  The model is trained on these news categories, not as a general fact checker.
  Try more training data, several seeds, or a longer input window; longer
  sequences also cost more because attention compares pairs of tokens.

### How to Explain This Project to My Professor

1. **Why an encoder only?** We read an entire article and predict one class.
   We need a representation of the input, not a decoder to generate words.
2. **How are Q, K, and V learned?** Three trainable linear layers project the
   same input token vectors into queries, keys, and values. Backpropagation
   from the news-label loss updates all three. No separate dataset is needed.
3. **How does self-attention work?** Each query scores the keys by dot product,
   divides by the square root of the head dimension, and uses softmax over the
   unmasked keys. The resulting weights mix the corresponding value vectors.
   A heatmap of those weights does not explain the whole decision.
4. **Why multiple heads?** Heads use different parts of the projections and
   can mix information in different ways. We join their outputs; no head is
   forced to learn a particular job.
5. **Why add positions?** Word order can change meaning. Without positional
   information, shuffling the retained words at inference does not tell this
   `[CLS]` classifier that their order changed.
6. **Why mask padding?** `[PAD]` cells only fill space. Masking their key scores
   before softmax gives them zero weight, so they cannot affect real tokens.
7. **How did we train from scratch?** We started with fresh trainable weights.
   We compared raw logits with integer class IDs using cross-entropy, then used
   gradients and AdamW to update the weights. Validation loss chose the epoch;
   the official test measured the chosen model afterward.
8. **What makes the experiments meaningful?** Each variant changes one setting
   from a fresh common reference and keeps data, seed, and budget matched.
   We compare on validation and report parameters and time. One seed cannot
   establish a general advantage.